# 0. Imports & initialisation Spark

In [1]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

# 0. Constantes globales du projet

In [2]:
#Chemin vers les données sur ma machine
CHEMIN_RAW = '../data/raw'
CHEMIN_INTERIM = '../data/interim'

#-------------------------------------------------------------#
# 13 codes paramètres SANDRE à conserver pour le pivot
parametres_codes = [
    # Contexte
    "1552",   # Volume moyen journalier
    "1553",   # Hauteur précipitations
    
    # Pollution organique
    "1313",   # DBO5
    "1314",   # DCO
    
    # Pollution particulaire
    "1305",   # MES
    
    # Pollution azotée (5 formes)
    "1319",   # NTK (Azote Kjeldahl)
    "1335",   # NH4 (Ammonium)
    "1339",   # NO2 (Nitrites)
    "1340",   # NO3 (Nitrates)
    "1551",   # Nglobal (Azote global)
    
    # Pollution phosphorée
    "1350",   # PT (Phosphore total)
    
    # Physico-chimie
    "1301",   # Température
    "1302",   # pH
]

#-------------------------------------------------------------#
# --- Colonnes-clés Pivot à conserver (ne seront PAS pivotées) ---
cles_ligne = [
    "CdSta",                # identifiant station
    "LbStaNo",              # nom station
    "DtPrel",               # date du prélèvement
    "AnPrel", "MoisPrel",   # temporel
    "NoDep",                # département
    "CdTypeSta", "LbTypeSta", # type d'ouvrage
    "CapNomSta",            # capacité nominale
]

#----------------------------------------------------------------#
# Dictionnaire de mapping code SANDRE → libellé court ML-friendly
mapping_libelles = {
    "1301": "Temperature",
    "1302": "pH",
    "1305": "MES",
    "1313": "DBO5",
    "1314": "DCO",
    "1319": "NTK",
    "1335": "NH4",
    "1339": "NO2",
    "1340": "NO3",
    "1350": "PT",
    "1551": "Nglobal",
    "1552": "Volume",
    "1553": "Pluie",
}

# 1. Chargement de données traitées

In [3]:
df_Filt = spark.read.parquet(CHEMIN_INTERIM + '/df_Filt.parquet')
print(f'df_Filt rechargé : {df_Filt.count():,} lignes')
df_Filt.printSchema()

df_Filt rechargé : 7,630,169 lignes
root
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- CdPtPrel: string (nullable = true)
 |-- CdSup: string (nullable = true)
 |-- LbSup: string (nullable = true)
 |-- CdFrac: string (nullable = true)
 |-- NmFrac: string (nullable = true)
 |-- CdPar: string (nullable = true)
 |-- LbPar: string (nullable = true)
 |-- RsAna: string (nullable = true)
 |-- SbUnt: string (nullable = true)



# 2. Pivot

In [4]:
# PréPivot : création des clés pivot
df_PrePivot = (
    df_Filt
    .withColumn('RsAna', sf.col('RsAna').cast('double'))
    .withColumn('ClPivot', sf.concat_ws('_',sf.col('CdPtPrel'),sf.col('CdPar')))
)

# Vérifications des clés pivot
print(df_PrePivot.select('ClPivot').distinct().count())
df_PrePivot.select('ClPivot').distinct().orderBy('ClPivot').show(5, truncate=False)

26


+-------+
|ClPivot|
+-------+
|A3_1301|
|A3_1302|
|A3_1305|
|A3_1313|
|A3_1314|
+-------+
only showing top 5 rows


In [5]:
# Pivot long
# --- Colonnes-clés à conserver (ne seront PAS pivotées) ---
cles_ligne = [
    "CdSta",                # identifiant station
    "NmNoSta",              # nom station
    "DtPrel",               # date du prélèvement
    "AnPrel", "MoPrel",   # temporel
    "NoDep",                # département
    "CdTpSta", "LbTpSta", # type d'ouvrage
    "CapNomSta",            # capacité nominale
]

df_Pivot = (
    df_PrePivot
    .groupBy(*cles_ligne)
    .pivot('ClPivot')
    .agg(sf.avg('RsAna'))
)

# --- Diagnostic ---
print(f'Nombre de lignes après pivot : {df_Pivot.count():,}')
print(f'Nombre de colonnes           : {len(df_Pivot.columns)}')
df_Pivot.printSchema()

df_Pivot.select(df_Pivot.columns[:10]).show(2, truncate=False)
df_Pivot.select(df_Pivot.columns[10:20]).show(2, truncate=False)
df_Pivot.select(df_Pivot.columns[20:30]).show(2, truncate=False)
df_Pivot.select(df_Pivot.columns[30:]).show(2, truncate=False)

Nombre de lignes après pivot : 2,024,836
Nombre de colonnes           : 35
root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- A3_1301: double (nullable = true)
 |-- A3_1302: double (nullable = true)
 |-- A3_1305: double (nullable = true)
 |-- A3_1313: double (nullable = true)
 |-- A3_1314: double (nullable = true)
 |-- A3_1319: double (nullable = true)
 |-- A3_1335: double (nullable = true)
 |-- A3_1339: double (nullable = true)
 |-- A3_1340: double (nullable = true)
 |-- A3_1350: double (nullable = true)
 |-- A3_1551: double (nullable = true)
 |-- A3_1552: double (nullable = true)
 |-- A3_1553: double (nullable = true)
 |-- A4_1301: double (nullable = true)
 |-- A4_1302: double (nullab

+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+-------+
|CdSta       |NmNoSta          |DtPrel    |AnPrel|MoPrel|NoDep|CdTpSta|LbTpSta                  |CapNomSta|A3_1301|
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+-------+
|030200201000|ACHERY__24920_029|04/05/2019|2019  |05    |02   |29     |STEP collectivité locale |50       |NULL   |
|030200201000|ACHERY__24920_029|13/03/2019|2019  |03    |02   |29     |STEP collectivité locale |50       |NULL   |
+------------+-----------------+----------+------+------+-----+-------+-------------------------+---------+-------+
only showing top 2 rows


+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|A3_1302|A3_1305|A3_1313|A3_1314|A3_1319|A3_1335|A3_1339|A3_1340|A3_1350|A3_1551|
+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |
|NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |
+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
only showing top 2 rows


+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|A3_1552|A3_1553|A4_1301|A4_1302|A4_1305|A4_1313|A4_1314|A4_1319|A4_1335|A4_1339|
+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|5.34   |5.2    |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |
|5.34   |3.4    |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |NULL   |
+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
only showing top 2 rows


+-------+-------+-------+-------+-------+
|A4_1340|A4_1350|A4_1551|A4_1552|A4_1553|
+-------+-------+-------+-------+-------+
|NULL   |NULL   |NULL   |5.34   |NULL   |
|NULL   |NULL   |NULL   |5.34   |NULL   |
+-------+-------+-------+-------+-------+
only showing top 2 rows


In [6]:
# Renommage automatique de toutes les colonnes pivotées
for col in df_Pivot.columns:
    # Est-ce une colonne pivotée (format : "1313_A3") ?
    if "_" in col and col.split("_")[1] in mapping_libelles:
        ValPtPrel, ValCdPar = col.split("_")
        col_nouv = f"{ValPtPrel}_{mapping_libelles[ValCdPar]}"
        df_Pivot = df_Pivot.withColumnRenamed(col, col_nouv)

df_Pivot.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [7]:
df_Pivot.write.mode('overwrite').parquet(CHEMIN_INTERIM + '/df_Pivot.parquet')
print("dataframe sauvegardé en Parquet")

dataframe sauvegardé en Parquet


In [8]:
spark.stop()